In [16]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('../data/dataset.csv')
df.head()

,area,body_type,brand,condition,engine_capacity,Fuel type:,km_run,listing_title,model,posted_on,price,reg_year,sold_by,subarea,transmission,additional_info,url,views,man_year
0,Dhaka\n1797\nviews,Saloon,Toyota,Used,"2,000 cc","CNG, Octane","116,000 km",Toyota Noah 2011,Noah,23/07/2025 12:56,"Tk 2,100,000",2016.0,Individual,Uttara,Automatic,NaN,https://bikroy.com/en/ad/toyota-noah-2011-for-...,1797.0,2011
1,Dhaka\n1171\nviews,MPV,Toyota,New,"1,800 cc","Petrol, Hybrid, Octane","35,200 km",Toyota Noah Z Hybrid Bronze 2022,Noah,23/07/2025 12:53,"Tk 4,350,000",NaN,Company,Baridhara,Automatic,Z Hybrid Bronze 2022,https://bikroy.com/en/ad/toyota-noah-z-hybrid-...,1171.0,2022
2,Dhaka\n9550\nviews,SUV / 4x4,SsangYong,Used,"1,500 cc","Petrol, Octane","20,000 km",SsangYong Korando 2020,Korando,23/07/2025 12:43,"Tk 3,300,000",2021.0,Individual,Gulshan,Automatic,NaN,https://bikroy.com/en/ad/ssangyong-korando-202...,9550.0,2020
3,Dhaka\n2794\nviews,MPV,Toyota,Used,"2,000 cc","CNG, Octane","66,000 km",Toyota Noah X 2010,Noah,23/07/2025 12:26,"Tk 2,050,000",2016.0,Individual,Dhanmondi,Automatic,X,https://bikroy.com/en/ad/toyota-noah-noha-x-20...,2794.0,2010
4,Dhaka\n9\nviews,Saloon,Toyota,Used,"1,500 cc","Petrol, CNG","88,000 km",Toyota Probox GL 2004,Probox,23/07/2025 12:22,"Tk 710,000",2008.0,Company,Mirpur,Automatic,GL,https://bikroy.com/en/ad/toyota-probox-gl-2004...,9.0,2004


In [17]:
df['price'] = df['price'].astype(str).str.replace('Tk ', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

df['km_run'] = df['km_run'].astype(str).str.replace(' km', '', regex=False)
df['km_run'] = df['km_run'].str.replace(',', '', regex=False)
df['km_run'] = pd.to_numeric(df['km_run'], errors='coerce')

df['engine_capacity'] = df['engine_capacity'].astype(str).str.replace(' cc', '', regex=False)
df['engine_capacity'] = df['engine_capacity'].str.replace(',', '', regex=False)
df['engine_capacity'] = pd.to_numeric(df['engine_capacity'], errors='coerce')

features_to_keep = ['brand', 'condition', 'reg_year', 'engine_capacity', 'km_run', 'price']
df = df[features_to_keep]

df = df.dropna()

print("Data is clean! Here are the new clean numbers:")
df.head()

Data is clean! Here are the new clean numbers:


,brand,condition,reg_year,engine_capacity,km_run,price
0,Toyota,Used,2016.0,2000,116000,2100000
2,SsangYong,Used,2021.0,1500,20000,3300000
3,Toyota,Used,2016.0,2000,66000,2050000
4,Toyota,Used,2008.0,1500,88000,710000
10,Toyota,Used,2011.0,1500,61530,1330000


In [18]:
X = df.drop('price', axis=1)
y = df['price']              

X = pd.get_dummies(X, columns=['brand', 'condition'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} cars, testing on {len(X_test)} cars.")

Training on 1833 cars, testing on 459 cars.


In [21]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

# 1. Initialize the 3 Models
models = {
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Linear Regression': LinearRegression()
}

saved_data = {
    'columns': X_train.columns.tolist(),
    'models': {},
    'stats': {}
}

# 2. Train and Evaluate All Models
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    # Calculate predictions
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    # Calculate stats
    saved_data['stats'][name] = {
        'r2_score': round(r2_score(y_test, test_preds) * 100, 2),
        'test_error_mae': round(mean_absolute_error(y_test, test_preds), 2)
    }
    saved_data['models'][name] = model

# 3. Save everything into ONE master file
with open('../models/multi_model.pkl', 'wb') as file:
    pickle.dump(saved_data, file)

print("All 3 models trained and saved successfully!")

Training Random Forest...
Training Gradient Boosting...
Training Linear Regression...
All 3 models trained and saved successfully!


In [23]:
import pickle

model_data = {
    'model': model,
    'columns': list(X.columns),
    'stats': {
        'r2_score': round(r2_test * 100, 2),
        'train_error_mae': round(train_mae, 2),
        'test_error_mae': round(test_mae, 2),
        'fit_status': fit_status
    }
}

with open('../models/car_price_model.pkl', 'wb') as file:
    pickle.dump(model_data, file)
    
print("Successfully saved the model AND the academic stats to car_price_model.pkl!")

Successfully saved the model AND the academic stats to car_price_model.pkl!


In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

r2_test = r2_score(y_test, test_predictions)

train_mae = mean_absolute_error(y_train, train_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)

train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print(f"Model R-Squared Score: {r2_test * 100:.2f}%")
print(f"Training Error (MAE): {train_mae:,.2f} Tk")
print(f"Testing Error (MAE): {test_mae:,.2f} Tk")

if train_mae < (test_mae * 0.5):
    fit_status = "Warning: Model might be OVERFITTING."
else:
    fit_status = "Model is well-fitted (Normal Variance)."
print(fit_status)

Model R-Squared Score: 61.78%
Training Error (MAE): 1,589,881.30 Tk
Testing Error (MAE): 1,608,851.97 Tk
Model is well-fitted (Normal Variance).
